In [1]:
import os

In [2]:
pwd

'c:\\Users\\adeka\\OneDrive\\Documents\\credit-scoring-Project\\research'

In [3]:
os.chdir(r"C:\Users\adeka\OneDrive\Documents\credit-scoring-Project")

print(os.getcwd())

C:\Users\adeka\OneDrive\Documents\credit-scoring-Project


In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    data_path: Path  
    all_schema: dict


In [5]:
from src.mlProject.constants import *
from src.mlProject.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path: Path = CONFIG_FILE_PATH,
        params_file_path: Path = PARAMS_FILE_PATH,
        schema_file_path: Path = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        create_directories([config.root_dir])

        return DataValidationConfig(
            root_dir=config.root_dir,
            STATUS_FILE=config.report_file,
            data_path=config.data_path,  
            all_schema=self.schema.COLUMNS
        )

In [7]:
import os
from src.mlProject.logging import logger
import pandas as pd
from src.mlProject.entity.config_entity import DataValidationConfig

class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_all_columns(self) -> bool:
        try:
            validation_status = None

            data = pd.read_csv(self.config.data_path, low_memory=False)
            all_cols = list(data.columns)
            all_schema = self.config.all_schema.keys()

            for col in all_cols:
                if col not in all_schema:
                    validation_status = False
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"validation_status: {validation_status}")
                else:
                    validation_status = True
                    with open(self.config.STATUS_FILE, 'w') as f:
                        f.write(f"validation_status: {validation_status}")

            return validation_status

        except Exception as e:
            raise e

In [8]:
from src.mlProject.logging import logger


STAGE_NAME = "Data Validation Stage"

class DataValidationTrainingPipeline:
    def __init__(self):
        pass

    def main(self):
        config = ConfigurationManager()
        data_validation_config = config.get_data_validation_config()

        data_validation = DataValidation(config=data_validation_config)

        data_validation.validate_all_columns()

if __name__ == "__main__":
    try:
        logger.info(f">>>>> stage {STAGE_NAME} started <<<<<")
        obj = DataValidationTrainingPipeline()
        obj.main()
        logger.info(f">>>>> stage {STAGE_NAME} completed <<<<<\n\nx==========x")
    except Exception as e:
        logger.exception(e)
        raise e

[2026-06-06 04:25:31,121: INFO: 464669567]
[2026-06-06 04:25:31,138: INFO: common]
[2026-06-06 04:25:31,143: INFO: common]
[2026-06-06 04:25:31,160: INFO: common]
[2026-06-06 04:25:31,168: INFO: common]
[2026-06-06 04:25:31,171: INFO: common]
[2026-06-06 04:25:34,219: INFO: 464669567]
